# ۳. تحلیل اکتشافی مزیت میزبانی

نرخ برد میزبان در هر دو منبع و مجموعه ادغام‌شده محاسبه و بر اساس فصل و مرحله مقایسه می‌شود.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DataSet2").is_dir())
ANALYSIS_ROOT = ROOT / "UCL_Analysis2"
OUTPUT = ANALYSIS_ROOT / "Output"
sys.path.insert(0, str(ANALYSIS_ROOT / "src"))
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")
print("Repository root:", ROOT)


In [ ]:
from scipy.stats import binomtest, chi2_contingency
from ucl_analysis import home_advantage_summary

detailed = pd.read_csv(OUTPUT / "detailed_matches_clean.csv")
ucl = pd.read_csv(OUTPUT / "ucl_matches_deduplicated.csv")
combined = pd.read_csv(OUTPUT / "combined_unique_matches.csv")

summary = pd.DataFrame([
    home_advantage_summary(detailed, "Dated 2005-2021"),
    home_advantage_summary(ucl, "UCL deduplicated 2010-2021"),
    home_advantage_summary(combined, "Combined unique matches"),
])
summary.to_csv(OUTPUT / "home_advantage_summary.csv", index=False)
display(summary.style.format({c: "{:.3%}" for c in ["home_win_rate", "home_win_ci_low", "home_win_ci_high", "draw_rate", "away_win_rate", "home_share_decisive"]}))


نرخ برد خام میزبان حدود ۴۷٪ است. اما چون مساوی هم یک نتیجه مستقل است، برای سنجش جهت مزیت، سهم برد میزبان در مسابقات غیرمساوی نیز محاسبه می‌شود؛ این مقدار نزدیک ۶۱٪ است.

In [ ]:
home_wins = int((combined.result == "H").sum())
away_wins = int((combined.result == "A").sum())
test = binomtest(home_wins, home_wins + away_wins, p=0.5, alternative="greater")
print(f"Home share among decisive matches: {home_wins/(home_wins+away_wins):.3%}")
print(f"Exact one-sided binomial p-value: {test.pvalue:.3e}")


In [ ]:
phase_table = pd.crosstab(detailed.phase, detailed.result)
chi2, p_value, dof, expected = chi2_contingency(phase_table)
phase_rates = detailed.groupby("phase").agg(
    matches=("result", "size"),
    home_win_rate=("home_win", "mean"),
    mean_goal_difference=("goal_difference", "mean"),
)
display(phase_table)
display(phase_rates.style.format({"home_win_rate": "{:.2%}", "mean_goal_difference": "{:.3f}"}))
print(f"Phase × result chi-square p-value: {p_value:.4g}")


In [ ]:
season = combined.groupby("season_start").agg(
    matches=("result", "size"), home_win_rate=("home_win", "mean"),
    mean_goal_difference=("goal_difference", "mean"),
).reset_index()
season.to_csv(OUTPUT / "home_advantage_by_season.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
result_rates = combined.result.value_counts(normalize=True).reindex(["H", "D", "A"])
axes[0].bar(["Home win", "Draw", "Away win"], result_rates.values * 100, color=["#2a9d8f", "#e9c46a", "#e76f51"])
axes[0].set_ylabel("Percent of matches")
axes[0].set_title("Combined result distribution")
for i, value in enumerate(result_rates.values * 100): axes[0].text(i, value + .5, f"{value:.1f}%", ha="center")

axes[1].plot(season.season_start, season.home_win_rate * 100, marker="o")
axes[1].axhline(combined.home_win.mean() * 100, color="black", ls="--", label="Overall")
axes[1].set_xlabel("Season start year")
axes[1].set_ylabel("Home-win rate (%)")
axes[1].set_title("Home-win rate by season")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT / "home_advantage_eda.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
home_team_stats = detailed.groupby("home_team").agg(
    home_matches=("home_win", "size"),
    home_wins=("home_win", "sum"),
    home_win_rate=("home_win", "mean"),
    mean_home_goal_difference=("goal_difference", "mean"),
).query("home_matches >= 10").sort_values(["home_win_rate", "home_matches"], ascending=False)
home_team_stats.to_csv(OUTPUT / "home_team_performance_min10.csv")
home_team_stats.head(15)
